# 03. Gaussian Process Regression for Robotics

Gaussian Process(GP)는 함수 자체에 대한 확률분포를 두는 비모수 Bayesian 모델이다.

$$f(x) \sim \mathcal{GP}(m(x), k(x,x'))$$

로보틱스에서는 terrain height, friction map, residual dynamics, calibration curve처럼 불확실한 연속 함수를 모델링할 때 유용하다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False


## 1. Kernel로 함수의 부드러움 표현하기

RBF kernel은 가까운 입력일수록 비슷한 출력값을 가진다는 prior를 표현한다.

$$k(x,x')=\sigma_f^2\exp\left(-\frac{(x-x')^2}{2\ell^2}\right)$$

In [ ]:
np.random.seed(503)

def true_f(x):
    return 0.45 * np.sin(1.7 * x) + 0.08 * x

def rbf_kernel(a, b, length=1.0, sigma_f=0.55):
    a = np.atleast_2d(a).T
    b = np.atleast_2d(b).T
    sqdist = (a - b.T) ** 2
    return sigma_f**2 * np.exp(-0.5 * sqdist / length**2)

X_train = np.array([-2.8, -1.7, -0.7, 0.2, 1.1, 2.4, 3.0])
y_train = true_f(X_train) + np.random.randn(len(X_train)) * 0.08
X_test = np.linspace(-3.3, 3.4, 240)
noise = 0.08

K = rbf_kernel(X_train, X_train) + noise**2 * np.eye(len(X_train))
Ks = rbf_kernel(X_train, X_test)
Kss = rbf_kernel(X_test, X_test) + 1e-9 * np.eye(len(X_test))
K_inv = np.linalg.inv(K)
mu = Ks.T @ K_inv @ y_train
cov = Kss - Ks.T @ K_inv @ Ks
std = np.sqrt(np.maximum(np.diag(cov), 0))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(X_test, true_f(X_test), 'k--', lw=1.5, label='true terrain/friction function')
ax.plot(X_test, mu, color='tab:blue', lw=2, label='GP posterior mean')
ax.fill_between(X_test, mu - 2*std, mu + 2*std, color='tab:blue', alpha=0.2, label='95% uncertainty')
ax.scatter(X_train, y_train, color='tab:red', zorder=3, label='samples')
ax.set_title('Gaussian Process regression with uncertainty')
ax.set_xlabel('robot position x')
ax.set_ylabel('unknown function value')
ax.legend()
plt.tight_layout()
plt.savefig('assets/stat_03_gp_regression.png', dpi=160)
plt.show()

## 2. Uncertainty로 다음 측정 위치 고르기

GP posterior variance가 큰 곳은 아직 모르는 영역이다. Exploration에서는 uncertainty가 큰 위치를 다음 측정 후보로 선택할 수 있다.

In [ ]:
next_idx = int(np.argmax(std))
next_x = X_test[next_idx]

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(X_test, std, color='tab:orange', lw=2)
ax.axvline(next_x, color='tab:red', ls='--', label=f'next sample x={next_x:.2f}')
ax.scatter(X_train, np.zeros_like(X_train), color='black', s=35, label='existing samples')
ax.set_title('Active sensing from GP posterior uncertainty')
ax.set_xlabel('robot position x')
ax.set_ylabel('posterior standard deviation')
ax.legend()
plt.tight_layout()
plt.savefig('assets/stat_03_gp_uncertainty_sampling.png', dpi=160)
plt.show()

print('next measurement candidate:', round(float(next_x), 3))

## 3. 로보틱스 연결

| 개념 | 의미 | 로보틱스 활용 |
|---|---|---|
| Kernel | 함수 prior의 smoothness/scale | terrain, friction, calibration curve |
| Posterior mean | 현재 데이터로 예측한 함수 | model compensation |
| Posterior variance | 예측 불확실성 | active sensing / exploration |
| Nonparametric model | 데이터가 늘면 표현력이 증가 | 복잡한 환경 모델링 |

GP는 계산량이 커질 수 있지만, 불확실성을 포함한 연속 모델을 만들 때 강력한 기준 모델이다.